# Exercise 4.2: Recreate a published plot

From *Programming in High Energy Particle Physics*, Chapter 4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/isildakbora/hep-programming-guide-notebooks/blob/main/ch04/ex04_2_solution.ipynb)

Exercise 4.2 Recreate a published plot Using Matplotlib and mplhep, recreate one of your favorite plots from a recent CMS or ATLAS paper. Pay attention to fonts, label positioning, and color choices.

<details><summary>Hint</summary>

The numerical content of most published plots is available on HEPData ( hepdata.net ) as tables that you can download and plot with hep.histplot . Start from hep.style.use("CMS") or hep.style.use("ATLAS") and compare side by side with the original.

</details>

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    get_ipython().run_line_magic("pip", "install -q numpy matplotlib mplhep")

### Recreate the published $p_T(H)$ spectrum

Source: [CMS HEPData record ins2931458, PT(H)](https://www.hepdata.net/record/ins2931458). This cell requires network access.

In [ ]:
import json, urllib.request, urllib.parse
import numpy as np
import matplotlib.pyplot as plt
import mplhep as hep
record='https://www.hepdata.net/record/ins2931458'
table='PT(H)'
url=record+'?'+urllib.parse.urlencode({'format':'json','table':table})
with urllib.request.urlopen(url,timeout=30) as response:
    table_data=json.load(response)
xrows=table_data['independent_variables'][0]['values']
yrows=table_data['dependent_variables'][0]['values']
edges=np.array([row['low'] for row in xrows]+[xrows[-1]['high']],dtype=float)
values=np.array([row['value'] for row in yrows],dtype=float)
def total_error(row):
    errs=[]
    for item in row.get('errors',[]):
        if 'symerror' in item: errs.append(abs(float(item['symerror'])))
        elif 'asymerror' in item:
            a=item['asymerror'];errs.append((abs(float(a['plus']))+abs(float(a['minus'])))/2)
    return np.hypot.reduce(errs) if errs else 0.0
errors=np.array([total_error(row) for row in yrows])
hep.style.use('CMS')
fig,ax=plt.subplots(figsize=(8,6))
hep.histplot(values,bins=edges,yerr=errors,histtype='errorbar',ax=ax,
             color='black',label='CMS published values')
ax.set_xlabel(r'$p_T(H)$ [GeV]')
unit=table_data['dependent_variables'][0].get('header',{}).get('units','as published')
ax.set_ylabel(r'$d\sigma/dp_T(H)$ ['+unit+']')
ax.set_yscale('log');ax.legend(frameon=False)
hep.cms.label(data=True,ax=ax)
ax.text(0.98,0.97,'HEPData ins2931458 · PT(H)',ha='right',va='top',transform=ax.transAxes,fontsize=9)
fig.savefig('published_recreation.png',dpi=150,bbox_inches='tight');plt.show()
print('Published bins:',len(values),'range:',edges[0],edges[-1])

In [ ]:
assert len(edges)==len(values)+1 and np.all(np.diff(edges)>0)

### Interpretation

The plotted values and uncertainties come from HEPData at run time. Visual comparison with the original figure still requires opening the linked publication. **Partial verification:** this environment cannot resolve HEPData, so the live table and its axis units could not be checked here.